In [1]:
import random
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import style
import matplotlib as mpl
import scanpy as sc
import anndata
import os.path as osp
from sklearn.metrics import adjusted_rand_score

In [2]:
import os
os.environ['CUDA_VISIBLE_DEVICES']='0'
import sys
from pathlib import Path

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
# GPU hardware acceleration
import torch 
device='cuda' if torch.cuda.is_available() else 'cpu'
%load_ext autoreload
%autoreload 2
import time
import spateo as st
import pyvista as pv
import pickle as pkl

Merfish mouse brain data

In [4]:

spatial_key = 'spatial'
cluster_key = 'original_clusters'
key_added = 'align_spatial'
# set random seeds
seeds_to_run=[2024,1,2,3,4,10,11,12,13,14]

in_ = '/home/data3/yuling/xenium_region2_5599/merfish_mouse_brain'
slice_ids_o = range(33)
sliceID =[[str(x) for x in slice_ids_o]]
end_no = 33
slices = [] #this is the original data
slice_ids = sliceID[0]
for i in range(len(slice_ids)):
    slice1 = sc.read(osp.join(in_, 'merfish_mouse_brain_slice'+slice_ids[i]+'.h5ad'))
    slice1.obs['original_clusters']=slice1.obs['spa_cluster'].astype('category')
    slice1.obsm['spatial'] = slice1.obsm['spatial'].to_numpy()[:, :2]
    slice1.obs['Z']=i
    slices.append(slice1)
    
for i in seeds_to_run:

    # create folder for each seed
    fold_path=osp.join('/home/data3/yuling/xenium_region2_5599/spateo_merfish', str(i))
    if not os.path.exists(fold_path):
        os.makedirs(fold_path)

    np.random.seed(i)
    torch.manual_seed(i)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(i)
        torch.cuda.manual_seed_all(i)  # If using multi-GPU
    aligned_slices, pis = st.align.morpho_align(
            models=slices,
            device=device,
            verbose=False,
            spatial_key=spatial_key,
            key_added=key_added,
            return_mapping=True
            )
    for cur_idx in range(1,len(slice_ids)):
        index=slice_ids[cur_idx-1]+'_'+slice_ids[cur_idx]
        map_path=osp.join(fold_path, index+'_AlignmentPi.pkl')
        with open(map_path,'wb') as f:
            pkl.dump(pis[cur_idx-1], f)

    aligned_adata=anndata.concat(aligned_slices)
    aligned_adata.obsm['spatial_3D']=np.concatenate([aligned_adata.obsm['align_spatial'],np.array(aligned_adata.obs['Z'].values)[:,None]],axis=1)
    df = pd.DataFrame(aligned_adata.obsm['spatial_3D'],
                  columns=['X', 'Y','Z'],
                  index=aligned_adata.obs_names.to_list())
    df['original_clusters']=aligned_adata.obs['original_clusters'].values
    df['slice']=aligned_adata.obs['slice_id'].values
    df.to_csv(osp.join(fold_path,'3d_coord_merfish.csv'))

|-----> [Models alignment based on morpho, mode: SN-S.] in progress: 100.0000%
|-----> [Models alignment based on morpho, mode: SN-S.] finished [41.2244s]
|-----> [Models alignment based on morpho, mode: SN-S.] in progress: 100.0000%
|-----> [Models alignment based on morpho, mode: SN-S.] finished [39.4821s]
|-----> [Models alignment based on morpho, mode: SN-S.] in progress: 100.0000%
|-----> [Models alignment based on morpho, mode: SN-S.] finished [41.5796s]
|-----> [Models alignment based on morpho, mode: SN-S.] in progress: 100.0000%
|-----> [Models alignment based on morpho, mode: SN-S.] finished [40.1657s]
|-----> [Models alignment based on morpho, mode: SN-S.] in progress: 100.0000%
|-----> [Models alignment based on morpho, mode: SN-S.] finished [42.4280s]
|-----> [Models alignment based on morpho, mode: SN-S.] in progress: 100.0000%
|-----> [Models alignment based on morpho, mode: SN-S.] finished [41.7105s]
|-----> [Models alignment based on morpho, mode: SN-S.] in progress: 1

simulated data

In [ ]:
spatial_key = 'spatial'
cluster_key = 'original_clusters'
key_added = 'align_spatial'

slice_ids =[151673,100,80,60,40,20]
seeds_to_run=[2024,1,2,3,4,10,11,12,13,14]
run_time_1_iter=0
run_time_2_iter=0
run_time_total_iter=0
    # load data and normalize
slices=[]
slice1=sc.read_h5ad('/home/data3/yuling/xenium_region2_5599/simData/DLPFC_151673original_spotIndex.h5')
slice1.layers["counts"] = slice1.X.copy()
sc.pp.normalize_total(slice1)
sc.pp.log1p(slice1)
slice1.obs['slice_id']=151673
slice1.obs['Z']=0
slice1.obs_names_make_unique()
slices.append(slice1)
for index,s in enumerate(slice_ids[1:]):
    slice1=sc.read_h5ad(osp.join('/home/data3/yuling/xenium_region2_5599/simData/DLPFC_151673_overlap='+str(s)+'__pseudocount_0_spotIndex.h5'))
    slice1.layers["counts"] = slice1.X.copy()
    sc.pp.normalize_total(slice1)
    sc.pp.log1p(slice1)
    slice1.obs['slice_id']=s
    slice1.obs['Z']=index+1
        
    slice1.obs_names_make_unique()
    slices.append(slice1) 
for i in seeds_to_run:

    # create folder for each seed
    fold_path=osp.join('/home/data3/yuling/xenium_region2_5599/simu_spateo', str(i))
    if not os.path.exists(fold_path):
        os.makedirs(fold_path)

    np.random.seed(i)

    torch.manual_seed(i) 
    if torch.cuda.is_available():
        torch.cuda.manual_seed(i)
        torch.cuda.manual_seed_all(i) 
    aligned_slices, pis = st.align.morpho_align(
            models=slices,
            device=device,
            verbose=False,
            spatial_key=spatial_key,
            key_added=key_added,
            return_mapping=True
            )
    for cur_idx in range(1,len(slice_ids)):
        index=str(slice_ids[cur_idx-1])+'_'+str(slice_ids[cur_idx])
        map_path=osp.join(fold_path, index+'_AlignmentPi.pkl')
        with open(map_path,'wb') as f:
            pkl.dump(pis[cur_idx-1], f)

    aligned_adata=anndata.concat(aligned_slices)
    aligned_adata.obsm['spatial_3D']=np.concatenate([aligned_adata.obsm['align_spatial'],np.array(aligned_adata.obs['Z'].values)[:,None]],axis=1)
    df = pd.DataFrame(aligned_adata.obsm['spatial_3D'],
                  columns=['X', 'Y','Z'],
                  index=aligned_adata.obs_names.to_list())
    df['original_clusters']=aligned_adata.obs['original_clusters'].values
    df['slice']=aligned_adata.obs['slice_id'].values
    df.to_csv(osp.join(fold_path,'3d_coord_simu.csv')) 


Mouse Hypothalamus data

In [ ]:
spatial_key = 'spatial'
cluster_key = 'original_clusters'
key_added = 'align_spatial'
slice_ids = ['-0.04', '-0.09',  '-0.14', '-0.19' , '-0.24']
seeds_to_run=[2024,1,2,3,4,10,11,12,13,14]
# load data and normalize
slices=[]
for j,section_ids in enumerate(slice_ids):
    slice1 = load_mHypothalamus(section_id=section_ids)
    slice1.var_names_make_unique()
    slice1.layers["counts"] = slice1.X.copy()
    sc.pp.normalize_total(slice1)
    sc.pp.log1p(slice1)
    slice1.obs['Z']=j
    slice1.obs['slice_id']=section_ids
    slices.append(slice1)
for i in seeds_to_run:

    # create folder for each seed
    fold_path=osp.join('/home/data3/yuling/xenium_region2_5599/spateo/mHy', str(i))
    if not os.path.exists(fold_path):
        os.makedirs(fold_path)
    np.random.seed(i)
    #random.seed(i) 
    torch.manual_seed(i) 
    if torch.cuda.is_available():
        torch.cuda.manual_seed(i)
        torch.cuda.manual_seed_all(i) 
    aligned_slices, pis = st.align.morpho_align(
            models=slices,
            device=device,
            verbose=False,
            spatial_key=spatial_key,
            key_added=key_added,
            return_mapping=True
            )
    for cur_idx in range(1,len(slice_ids)):
        index=slice_ids[cur_idx-1]+'_'+slice_ids[cur_idx]
        map_path=osp.join(fold_path, index+'_AlignmentPi.pkl')
        with open(map_path,'wb') as f:
            pkl.dump(pis[cur_idx-1], f)
    
    aligned_adata=anndata.concat(aligned_slices)
    aligned_adata.obsm['spatial_3D']=np.concatenate([aligned_adata.obsm['align_spatial'],np.array(aligned_adata.obs['Z'].values)[:,None]],axis=1)
    df = pd.DataFrame(aligned_adata.obsm['spatial_3D'],
                columns=['X', 'Y','Z'],
                index=aligned_adata.obs_names.to_list())
    
    df['original_clusters']=aligned_adata.obs['original_clusters'].values
    df['slice']=aligned_adata.obs['slice_id'].values
    df.to_csv(osp.join(fold_path,'3d_coord_mHy.csv'))



DLPFC data

In [ ]:
spatial_key = 'spatial'
cluster_key = 'original_clusters'
key_added = 'align_spatial'
# ['151669', '151670',  '151671', '151672']
# ['151673', '151674',  '151675', '151676']
slice_ids = ['151507', '151508',  '151509', '151510']
seeds_to_run=[2024,1,2,3,4,10,11,12,13,14]

for i in seeds_to_run:
    # load data and normalize
    slices=[]
    for j,section_ids in enumerate(slice_ids):
        slice1 = load_DLPFC(section_id=section_ids)
        slice1.layers["counts"] = slice1.X.copy()
        sc.pp.normalize_total(slice1)
        sc.pp.log1p(slice1)
        slice1.obs['Z']=j
        slice1.obs['slice_id']=section_ids
        
        slices.append(slice1)
    # create folder for each seed
    fold_path=osp.join('/home/data3/yuling/xenium_region2_5599/spateo/DLPFC_4/151507_151510', str(i))
    if not os.path.exists(fold_path):
        os.makedirs(fold_path)

    np.random.seed(i)
    torch.manual_seed(i)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(i)
        torch.cuda.manual_seed_all(i)  # If using multi-GPU
    aligned_slices, pis = st.align.morpho_align(
            models=slices,
            device=device,
            verbose=False,
            spatial_key=spatial_key,
            key_added=key_added,
            return_mapping=True
            )
    for cur_idx in range(1,len(slice_ids)):
        index=slice_ids[cur_idx-1]+'_'+slice_ids[cur_idx]
        map_path=osp.join(fold_path, index+'_AlignmentPi.pkl')

        with open(map_path,'wb') as f:
            pkl.dump(pis[cur_idx-1], f)
#### save coordinates
    aligned_adata=anndata.concat(aligned_slices)
    aligned_adata.obsm['spatial_3D']=np.concatenate([aligned_adata.obsm['align_spatial'],np.array(aligned_adata.obs['Z'].values)[:,None]],axis=1)
    df = pd.DataFrame(aligned_adata.obsm['spatial_3D'],
                  columns=['X', 'Y','Z'],
                  index=aligned_adata.obs_names.to_list())

    df['original_clusters']=aligned_adata.obs['original_clusters'].values
    df['slice']=aligned_adata.obs['slice_id'].values
    df.to_csv(osp.join(fold_path,'3d_coord_151507_151510_1.csv'))


MOSTA embryo data

In [ ]:
spatial_key = 'spatial'
cluster_key = 'original_clusters'
key_added = 'align_spatial'
slice_ids = ['E11.5_E1S1.MOSTA', 'E11.5_E1S2.MOSTA',  'E11.5_E1S3.MOSTA', 'E11.5_E1S4.MOSTA']
seeds_to_run=[2024,1,2,3,4,10,11,12,13,14]
# load data and normalize
slices=[]
for j,section_ids in enumerate(slice_ids):
    slice1 = load_Mostaembryo(section_id=section_ids)
        #slice1.layers["counts"] = slice1.X.copy()
    sc.pp.normalize_total(slice1)
    sc.pp.log1p(slice1)
    slice1.obs['Z']=j
    slice1.obs['slice_id']=section_ids
    slices.append(slice1)

for i in seeds_to_run:

    # create folder for each seed
    fold_path=osp.join('/home/data3/yuling/xenium_region2_5599/spateo_align/mosta_embryo', str(i))
    if not os.path.exists(fold_path):
        os.makedirs(fold_path)
    np.random.seed(i)
    #random.seed(i) 
    torch.manual_seed(i) 
    if torch.cuda.is_available():
        torch.cuda.manual_seed(i)
        torch.cuda.manual_seed_all(i) 
    aligned_slices, pis = st.align.morpho_align(
            models=slices,
            device=device,
            verbose=False,
            spatial_key=spatial_key,
            key_added=key_added,
            return_mapping=True
            )
    for cur_idx in range(1,len(slice_ids)):
        index=slice_ids[cur_idx-1]+'_'+slice_ids[cur_idx]
        map_path=osp.join(fold_path, index+'_AlignmentPi.pkl')
        with open(map_path,'wb') as f:
            pkl.dump(pis[cur_idx-1], f)
    
    aligned_adata=anndata.concat(aligned_slices)
    aligned_adata.obsm['spatial_3D']=np.concatenate([aligned_adata.obsm['align_spatial'],np.array(aligned_adata.obs['Z'].values)[:,None]],axis=1)
    df = pd.DataFrame(aligned_adata.obsm['spatial_3D'],
                columns=['X', 'Y','Z'],
                index=aligned_adata.obs_names.to_list())
    
    df['original_clusters']=aligned_adata.obs['original_clusters'].values
    df['slice']=aligned_adata.obs['slice_id'].values
    df.to_csv(osp.join(fold_path,'3d_coord_mosta_embryo.csv'))
